In [ ]:
# import all required libraries
import sys, os
import numpy as np
import pandas as pd
import random
from random import shuffle, choice
import time
import os
import glob
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.models import load_model
from tensorflow.keras import regularizers
from random import shuffle, choice
from sklearn.preprocessing import MinMaxScaler
import sklearn.metrics as metrics
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Model
from sklearn.preprocessing import MinMaxScaler,StandardScaler

# define a function to build a CNN for the SNP data.
def create_cnn(xtest, regularizer=None):
  # obtain the input dimensions.
  inputShape = (xtest.shape[1], xtest.shape[2])
  inputs = Input(shape=inputShape)
  x = inputs
  # first convolutional layer, remember to remove bias if you are intercalating with batch normalization.
  x = Conv1D(256, kernel_size=3, activation='relu', use_bias=False)(x)
  # batch normalization.
  x = BatchNormalization()(x)
  # second layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # third layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # pool the CNN outputs.
  x = GlobalMaxPooling1D()(x)
  # this part is similar to the MLP, a fully connected neural network. We intercalated with dropout to reduce overfitting.
  x = Dense(128, activation='relu')(x)
  # dropout.
  x = Dropout(0.5)(x)
  # second layer of the fully connected neural network.
  x = Dense(128, activation='relu')(x)
  x = Dropout(0.5)(x)
  # third layer of the fully connected neural network. This one matches the number of nodes coming out of the MLP.
  x = Dense(64, activation='relu')(x)
  # Construct the CNN
  #x = BatchNormalization()(x)#Not working very well
  #x = LayerNormalization()(x)#Better?
  model = Model(inputs, x)
  # Return the CNN
  return model

class GatedConcatenate(Layer):
    """
    Applies a trainable or fixed gate (weight) to each input branch
    before concatenating them.
    
    Args:
        initial_traits_weight (float): The starting weight for the first input (traits).
                                     Must be between 0 and 1. The weight for the
                                     second input (SNPs) will be (1 - this value).
        trainable_gates (bool): If True, the model can learn to adjust these
                                weights. If False, the weights are fixed.
    """
    def __init__(self, initial_traits_weight, trainable_gates=True, **kwargs):
        super(GatedConcatenate, self).__init__(**kwargs)
        if not (0 <= initial_traits_weight <= 1):
            raise ValueError("initial_traits_weight must be between 0 and 1.")
            
        self.initial_weights = [initial_traits_weight, 1.0 - initial_traits_weight]
        self.trainable_gates = trainable_gates

    def build(self, input_shape):
        # Create the gate variables. They are shaped for broadcasting across the features.
        self.gates = self.add_weight(
            name='gates',
            shape=(1, len(input_shape)), # Shape will be (1, 2)
            initializer=tf.constant_initializer(self.initial_weights),
            trainable=self.trainable_gates,
            dtype=tf.float32
        )
        super(GatedConcatenate, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, list) or len(inputs) != 2:
            raise ValueError("GatedConcatenate expects a list of exactly two input tensors.")
        
        # Apply the gates (weights) to each branch using element-wise multiplication
        gated_traits = inputs[0] * self.gates[0, 0]
        gated_snps = inputs[1] * self.gates[0, 1]
        
        # Concatenate the scaled branches
        return Concatenate()([gated_traits, gated_snps])

    def get_config(self):
        # Needed for saving/loading the model
        config = super().get_config()
        config.update({
            'initial_traits_weight': self.initial_weights[0],
            'trainable_gates': self.trainable_gates,
        })
        return config
    
def gated_contributions(model, layer_name=None, labels=("traits", "SNPs")):
    # 1) find the layer
    gated_layer = model.get_layer('gated_concatenate')
    weights = gated_layer.get_weights()[0][0]
    rel_weight = np.sum(np.abs(weights))
    print(f"Final learned weights: Traits={weights[0]/rel_weight:.4f}, SNPs={weights[1]/rel_weight:.4f}")

In [ ]:
# The Lepomis species complex dataset is much larger than the other datasets analyzed. 
# We modified the script to perform the same approach as in the other notebooks, but using mempory in a more efficient way.
# Beacuse of that, this notebook is less didactic then the others. So we recommend using the other notebooks for less advanced users.
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Enabled memory growth on GPUs.")
    except Exception as e:
        print("Could not set memory growth:", e)

Enabled memory growth on GPUs.


In [3]:
## define variables that will be used to train all networks.
# size of the minibatches containing simulations are passed through the network in each epoch.
batch_size = 256
# number of training iterations (epochs) for the SNP only and the combined networks.
epochs = 100
# number of training iterations (epochs) for the traits only networks.
num_classes = 4

In [ ]:
################################################################################################################################################
#Load data
################################################################################################################################################

traits_BM = []
traits_BM = np.loadtxt("./traits/traits_BM.txt").reshape(40000,-1,28)
#traits_BM=traits_BM[0:30000]

scalers_BM = {}
for i in range(traits_BM.shape[2]):
    scalers_BM[i] = StandardScaler(copy=False)
    traits_BM[:, :, i] = scalers_BM[i].fit_transform(traits_BM[:, :, i]) 

traits_BM = np.array(traits_BM)

#Add missing individuals
for i in range(traits_BM.shape[0]):
  for j in range(0,4):
    traits_BM[i,j,:] = 0
  for j in range(77,81):
    traits_BM[i,j,:] = 0
  for j in range(134,136):
    traits_BM[i,j,:] = 0

d_traits = []
d_traits = np.loadtxt("./traits/traits_disc.txt").reshape(40000,-1,3)

traits = np.concatenate((traits_BM,d_traits),axis=2)

In [ ]:
u1 = np.load("./trainingSims/Model_2sp.npz")
u2 = np.load("./trainingSims/Model_6sp.npz")
u3 = np.load("./trainingSims/Model_6spMig.npz")
u4 = np.load("./trainingSims/Model_6spMig_all.npz")

y=[0 for i in range(len(u1['Model_2sp']))]
y.extend([1 for i in range(len(u2['Model_6sp']))])
y.extend([2 for i in range(len(u3['Model_6spMig']))])
y.extend([3 for i in range(len(u4['Model_6spMig_all']))])
y = np.array(y)

# Extract arrays
a1 = u1['Model_2sp']
a2 = u2['Model_6sp']
a3 = u3['Model_6spMig']
a4 = u4['Model_6spMig_all']

def transform_major_minor(x):
    """
    Thus is a memory-efficient funciton to transform major alleles -> -1, minor alleles -> 1 (per site).
    """
    for arr_idx, array in enumerate(x):
        for row_idx, row in enumerate(array):
            # if more than half are 1s, 1 is major
            if np.count_nonzero(row) > len(row) / 2:
                x[arr_idx][row_idx][x[arr_idx][row_idx] == 1] = -1
                x[arr_idx][row_idx][x[arr_idx][row_idx] == 0] = 1
            else:
                x[arr_idx][row_idx][x[arr_idx][row_idx] == 0] = -1
    return x

# Apply separately
a1 = transform_major_minor(a1)
a2 = transform_major_minor(a2)
a3 = transform_major_minor(a3)
a4 = transform_major_minor(a4)

#Add missing data as 0s, according to a specifies missing data percentage
#54,477,268 SNP datapoints and 24,326,002 missing genotypes = 44.6%
missD_perc = 44.6

def add_missing_vectorized(arr, miss_perc, rng=None):
    """
    This is also a memor-efficient function.
    It replaces a percentage of entries with 0 (simulate missing genotypes).
    Operates in-place; returns the same array for convenience.

    Parameters
    ----------
    arr : np.ndarray
        Array of shape (n_reps, n_snps, n_samples)
    miss_perc : float
        Percentage (0-100) of entries to set missing per replicate
    rng : np.random.Generator, optional
        NumPy random generator for reproducibility
    """
    if rng is None:
        rng = np.random.default_rng()

    n_reps, n_snps, n_samples = arr.shape
    total_sites = n_snps * n_samples
    miss_count = int(total_sites * (miss_perc / 100.0))
    if miss_count <= 0:
        return arr

    for i in range(n_reps):
        flat = arr[i].reshape(-1)  # view (no copy)
        if miss_count >= total_sites:
            flat[:] = 0
            continue
        idx = rng.choice(total_sites, size=miss_count, replace=False)
        flat[idx] = 0

    return arr

# Apply missing data separately to each dataset
a1 = add_missing_vectorized(a1, missD_perc, rng=None)
a2 = add_missing_vectorized(a2, missD_perc, rng=None)
a3 = add_missing_vectorized(a3, missD_perc, rng=None)
a4 = add_missing_vectorized(a4, missD_perc, rng=None)

x=np.concatenate((a1,a2,a3,a4),axis=0)

del(a1,a2,a3,a4,u1,u2,u3,u4)

# Save results, to load more efficiently
np.save("./tmp/x_BM.npy", x)
np.save("./tmp/traits_BM.npy", traits)
np.save("./tmp/y_BM.npy", y)

In [ ]:
# open memmaps (memory efficient)
x_all_mm = np.load("./tmp/x_BM.npy", mmap_mode='r')          # shape (N, s1, s2)
traits_all_mm = np.load("./tmp/traits_BM.npy", mmap_mode='r')# shape (N, t1, t2)
y_all_mm = np.load("./tmp/y_BM.npy", mmap_mode='r')          # shape (N,)

In [ ]:
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
# 1) Make an index array and split indices (cheap)
n = y_all_mm.shape[0]   # number of examples
indices = np.arange(n)

train_idx, test_idx = train_test_split(
    indices, test_size=0.25, shuffle=True, stratify=y_all_mm
)

In [8]:
# generator using indices array
def index_generator_from_idx(idx_array, shuffle=False, seed=42):
    rng = np.random.RandomState(seed)
    indices = idx_array.copy()
    if shuffle:
        rng.shuffle(indices)
    for i in indices:
        traits_row = np.swapaxes(traits_all_mm[i], 0, 1).astype(np.float32, copy=False)
        snps_row   = x_all_mm[i].astype(np.float32)
        label      = int(y_all_mm[i])  # scalar
        yield (traits_row, snps_row), label
        
# set shapes for output_signature
traits_shape = tuple(np.swapaxes(np.empty(traits_all_mm.shape[1:]), 0, 1).shape)
snps_shape   = x_all_mm.shape[1:]

output_sig = (
    (tf.TensorSpec(shape=traits_shape, dtype=tf.float32),
     tf.TensorSpec(shape=snps_shape, dtype=tf.float32)),
    tf.TensorSpec(shape=(), dtype=tf.int64)
)

def make_index_dataset_from_idx(idx_array, batch_size=8, shuffle=False, prefetch=1):
    gen = lambda: index_generator_from_idx(idx_array, shuffle=shuffle)
    ds = tf.data.Dataset.from_generator(gen, output_signature=output_sig)
    # Keep shuffle buffer small (we already shuffled indices earlier if desired)
    if shuffle:
        ds = ds.shuffle(buffer_size=256)
    # Use drop_remainder=True to keep shapes constant (helps performance)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(prefetch)
    return ds

# Build datasets
train_ds = make_index_dataset_from_idx(train_idx, batch_size=256, shuffle=True, prefetch=1)
val_ds   = make_index_dataset_from_idx(test_idx,  batch_size=256, shuffle=False, prefetch=1)

In [ ]:
# This is the same as in the other notebooks, just simplified, as we now need this to be more memory efficient.
def combined_BM_subset(xtrain, traits_BM_train, train_ds, val_ds):
    # Create the two CNNs and the combined models
    traits = create_cnn(np.swapaxes(traits_BM_train, 1, 2))
    snps = create_cnn(xtrain)

    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss='sparse_categorical_crossentropy',
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit(train_ds,
          epochs=epochs,
          verbose=1,
          validation_data=val_ds,callbacks=[earlyStopping])
    
    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')

    return model

# function to train on the SNP only datasets
def SNP_subset(xtrain, train_ds, val_ds):
    # Create the the CNN 
    snps = create_cnn(xtrain)
    
    #Create the last layer for the SNP network
    xSNP = Dense(num_classes, activation="softmax")(snps.output)
    model = Model(inputs=snps.input, outputs=xSNP)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss='sparse_categorical_crossentropy',
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit(train_ds,
          epochs=epochs,
          verbose=1,
          validation_data=val_ds,callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the BM trait only datasets
def BM_subset(traits_BM_train, train_ds, val_ds):
    trait = create_cnn(np.swapaxes(traits_BM_train, 1, 2))
    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss='sparse_categorical_crossentropy',
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(train_ds,
          epochs=epochs,
          verbose=1,
          validation_data=val_ds,callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')

    return model

In [ ]:
################################################################################################################################################
#Train the combined SNPs + traits (BM) network
################################################################################################################################################

# train the network
model1 = combined_BM_subset(x_all_mm, traits_all_mm, train_ds, val_ds)

model1.save(filepath='./TrainedModels/Trained_Comb_Model_1KSNPs_BM.acc.mod')

Model: "model_5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 31, 229)]    0                                            
__________________________________________________________________________________________________
input_4 (InputLayer)            [(None, 1000, 458)]  0                                            
__________________________________________________________________________________________________
conv1d_6 (Conv1D)               (None, 31, 256)      175872      input_3[0][0]                    
__________________________________________________________________________________________________
conv1d_9 (Conv1D)               (None, 1000, 256)    351744      input_4[0][0]                    
____________________________________________________________________________________________

Epoch 69/100
117/117 [==============================] - 65s 550ms/step - loss: 0.0042 - accuracy: 0.9988 - val_loss: 0.4906 - val_accuracy: 0.9250
Epoch 70/100
117/117 [==============================] - 63s 538ms/step - loss: 0.0042 - accuracy: 0.9990 - val_loss: 0.4342 - val_accuracy: 0.9324
Epoch 71/100
117/117 [==============================] - 62s 529ms/step - loss: 0.0050 - accuracy: 0.9985 - val_loss: 0.4435 - val_accuracy: 0.9305
Epoch 72/100
117/117 [==============================] - 62s 525ms/step - loss: 0.0042 - accuracy: 0.9987 - val_loss: 0.4169 - val_accuracy: 0.9339
Epoch 73/100
117/117 [==============================] - 63s 533ms/step - loss: 0.0034 - accuracy: 0.9991 - val_loss: 0.4445 - val_accuracy: 0.9313
Epoch 74/100
117/117 [==============================] - 65s 548ms/step - loss: 0.0036 - accuracy: 0.9990 - val_loss: 0.4178 - val_accuracy: 0.9350
Epoch 75/100
117/117 [==============================] - 64s 544ms/step - loss: 0.0033 - accuracy: 0.9991 - val_loss: 0

In [10]:
################################################################################################################################################
#Train the SNPs only network
################################################################################################################################################
# keep only SNP input
train_snps_ds = train_ds.map(lambda inputs, y: (inputs[1], y))
val_snps_ds   = val_ds.map(lambda inputs, y: (inputs[1], y))

# train the network
model2 = SNP_subset(x_all_mm, train_snps_ds, val_snps_ds)

model2.save(filepath='./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod')


Model: "model_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 1000, 458)]       0         
_________________________________________________________________
conv1d (Conv1D)              (None, 1000, 256)         351744    
_________________________________________________________________
batch_normalization (BatchNo (None, 1000, 256)         1024      
_________________________________________________________________
conv1d_1 (Conv1D)            (None, 1000, 256)         196608    
_________________________________________________________________
batch_normalization_1 (Batch (None, 1000, 256)         1024      
_________________________________________________________________
conv1d_2 (Conv1D)            (None, 1000, 256)         196608    
_________________________________________________________________
batch_normalization_2 (Batch (None, 1000, 256)         1024

Epoch 42/100
117/117 [==============================] - 62s 529ms/step - loss: 0.0330 - accuracy: 0.9896 - val_loss: 0.2548 - val_accuracy: 0.9423
Epoch 43/100
117/117 [==============================] - 63s 537ms/step - loss: 0.0292 - accuracy: 0.9904 - val_loss: 0.2628 - val_accuracy: 0.9406
Epoch 44/100
117/117 [==============================] - 64s 542ms/step - loss: 0.0263 - accuracy: 0.9913 - val_loss: 0.2644 - val_accuracy: 0.9423
Epoch 45/100
117/117 [==============================] - 64s 542ms/step - loss: 0.0278 - accuracy: 0.9907 - val_loss: 0.2597 - val_accuracy: 0.9397
Epoch 46/100
117/117 [==============================] - 65s 550ms/step - loss: 0.0278 - accuracy: 0.9908 - val_loss: 0.2592 - val_accuracy: 0.9436
Epoch 47/100
117/117 [==============================] - 63s 532ms/step - loss: 0.0254 - accuracy: 0.9917 - val_loss: 0.2739 - val_accuracy: 0.9421
Epoch 48/100
117/117 [==============================] - 62s 526ms/step - loss: 0.0268 - accuracy: 0.9912 - val_loss: 0

Epoch 98/100
117/117 [==============================] - 62s 529ms/step - loss: 0.0118 - accuracy: 0.9961 - val_loss: 0.3354 - val_accuracy: 0.9467
Epoch 99/100
117/117 [==============================] - 64s 541ms/step - loss: 0.0123 - accuracy: 0.9962 - val_loss: 0.3052 - val_accuracy: 0.9495
Epoch 100/100
117/117 [==============================] - 64s 539ms/step - loss: 0.0119 - accuracy: 0.9962 - val_loss: 0.3215 - val_accuracy: 0.9496
Time: 6622.5344886779785
INFO:tensorflow:Assets written to: ./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod/assets


In [11]:
################################################################################################################################################
#Train the traits only (BM) network
################################################################################################################################################
train_traits_ds = train_ds.map(lambda inputs, y: (inputs[0], y))
val_traits_ds   = val_ds.map(lambda inputs, y: (inputs[0], y))

# train the network
model3 = BM_subset(traits_all_mm, train_traits_ds, val_traits_ds)
          
model3.save(filepath='./TrainedModels/Trained_Traits_Model_BM.acc.mod')

Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 31, 229)]         0         
_________________________________________________________________
conv1d_3 (Conv1D)            (None, 31, 256)           175872    
_________________________________________________________________
batch_normalization_3 (Batch (None, 31, 256)           1024      
_________________________________________________________________
conv1d_4 (Conv1D)            (None, 31, 256)           196608    
_________________________________________________________________
batch_normalization_4 (Batch (None, 31, 256)           1024      
_________________________________________________________________
conv1d_5 (Conv1D)            (None, 31, 256)           196608    
_________________________________________________________________
batch_normalization_5 (Batch (None, 31, 256)           1024

Epoch 42/100
117/117 [==============================] - 63s 540ms/step - loss: 0.0335 - accuracy: 0.9885 - val_loss: 0.3418 - val_accuracy: 0.9249
Epoch 43/100
117/117 [==============================] - 63s 536ms/step - loss: 0.0333 - accuracy: 0.9890 - val_loss: 0.3322 - val_accuracy: 0.9261
Epoch 44/100
117/117 [==============================] - 64s 545ms/step - loss: 0.0318 - accuracy: 0.9890 - val_loss: 0.3428 - val_accuracy: 0.9248
Epoch 45/100
117/117 [==============================] - 63s 539ms/step - loss: 0.0301 - accuracy: 0.9898 - val_loss: 0.3616 - val_accuracy: 0.9247
Epoch 46/100
117/117 [==============================] - 65s 551ms/step - loss: 0.0271 - accuracy: 0.9912 - val_loss: 0.3528 - val_accuracy: 0.9246
Epoch 47/100
117/117 [==============================] - 63s 541ms/step - loss: 0.0289 - accuracy: 0.9904 - val_loss: 0.3492 - val_accuracy: 0.9263
Epoch 48/100
117/117 [==============================] - 64s 547ms/step - loss: 0.0242 - accuracy: 0.9920 - val_loss: 0

Epoch 98/100
117/117 [==============================] - 65s 554ms/step - loss: 0.0079 - accuracy: 0.9974 - val_loss: 0.4838 - val_accuracy: 0.9302
Epoch 99/100
117/117 [==============================] - 65s 553ms/step - loss: 0.0086 - accuracy: 0.9971 - val_loss: 0.4860 - val_accuracy: 0.9308
Epoch 100/100
117/117 [==============================] - 64s 550ms/step - loss: 0.0086 - accuracy: 0.9968 - val_loss: 0.4924 - val_accuracy: 0.9300
Time: 6387.271549701691
INFO:tensorflow:Assets written to: ./TrainedModels/Trained_Traits_Model_BM.acc.mod/assets


In [ ]:
################################################################################################################################################
#Load OU traits
################################################################################################################################################

traits_OU = []
traits_OU = np.loadtxt("./traits/traits_OU.txt").reshape(40000,-1,28)

scalers_OU = {}
for i in range(traits_OU.shape[2]):
    scalers_OU[i] = StandardScaler(copy=False)
    traits_OU[:, :, i] = scalers_OU[i].fit_transform(traits_OU[:, :, i]) 

traits_OU = np.array(traits_OU)

#Add missing individuals
for i in range(traits_OU.shape[0]):
  for j in range(0,4):
    traits_OU[i,j,:] = 0
  for j in range(77,81):
    traits_OU[i,j,:] = 0
  for j in range(134,136):
    traits_OU[i,j,:] = 0

d_traits = []
d_traits = np.loadtxt("./traits/traits_disc.txt").reshape(40000,-1,3)

traits = np.concatenate((traits_OU,d_traits),axis=2)

In [ ]:
# Save results
np.save("./tmp/traits_OU.npy", traits)
del(traits)
traits_all_mm = np.load("./tmp/traits_OU.npy", mmap_mode='r')# shape (N, t1, t2)

In [14]:
# generator using indices array
def index_generator_from_idx(idx_array, shuffle=False, seed=42):
    rng = np.random.RandomState(seed)
    indices = idx_array.copy()
    if shuffle:
        rng.shuffle(indices)
    for i in indices:
        traits_row = np.swapaxes(traits_all_mm[i], 0, 1).astype(np.float32, copy=False)
        snps_row   = x_all_mm[i].astype(np.float32)
        label      = int(y_all_mm[i])  # scalar
        yield (traits_row, snps_row), label
        
# set shapes for output_signature
traits_shape = tuple(np.swapaxes(np.empty(traits_all_mm.shape[1:]), 0, 1).shape)
snps_shape   = x_all_mm.shape[1:]

output_sig = (
    (tf.TensorSpec(shape=traits_shape, dtype=tf.float32),
     tf.TensorSpec(shape=snps_shape, dtype=tf.float32)),
    tf.TensorSpec(shape=(), dtype=tf.int64)
)

def make_index_dataset_from_idx(idx_array, batch_size=8, shuffle=False, prefetch=1):
    gen = lambda: index_generator_from_idx(idx_array, shuffle=shuffle)
    ds = tf.data.Dataset.from_generator(gen, output_signature=output_sig)
    # Keep shuffle buffer small (we already shuffled indices earlier if desired)
    if shuffle:
        ds = ds.shuffle(buffer_size=256)
    # Use drop_remainder=True to keep shapes constant (helps performance)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(prefetch)
    return ds

# Build datasets
train_ds = make_index_dataset_from_idx(train_idx, batch_size=256, shuffle=True, prefetch=1)
val_ds   = make_index_dataset_from_idx(test_idx,  batch_size=256, shuffle=False, prefetch=1)

In [ ]:
def combined_OU_subset(xtrain, traits_OU_train, train_ds, val_ds):
    # Create the two CNNs and the combined models
    traits = create_cnn(np.swapaxes(traits_OU_train, 1, 2))
    snps = create_cnn(xtrain)

    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss='sparse_categorical_crossentropy',
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()

    model.fit(train_ds,
          epochs=epochs,
          verbose=1,
          validation_data=val_ds,callbacks=[earlyStopping])
    
    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')
    return model

# function to train on the SNP only datasets
def SNP_subset(xtrain, train_ds, val_ds):
    # Create the the CNN 
    snps = create_cnn(xtrain)
    
    #Create the last layer for the SNP network
    xSNP = Dense(num_classes, activation="softmax")(snps.output)
    model = Model(inputs=snps.input, outputs=xSNP)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss='sparse_categorical_crossentropy',
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit(train_ds,
          epochs=epochs,
          verbose=1,
          validation_data=val_ds,callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the BM trait only datasets
def OU_subset(traits_OU_train, train_ds, val_ds):
    trait = create_cnn(np.swapaxes(traits_OU_train, 1, 2))
    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss='sparse_categorical_crossentropy',
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(train_ds,
          epochs=epochs,
          verbose=1,
          validation_data=val_ds,callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')

    return model

In [ ]:
################################################################################################################################################
#Train the combined SNPs + traits (OU) network
################################################################################################################################################

# train the network
model4 = combined_OU_subset(x_all_mm, traits_all_mm, train_ds, val_ds)

model4.save(filepath='./TrainedModels/Trained_Comb_Model_1KSNPs_OU.acc.mod')

Model: "model_8"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_5 (InputLayer)            [(None, 31, 229)]    0                                            
__________________________________________________________________________________________________
input_6 (InputLayer)            [(None, 1000, 458)]  0                                            
__________________________________________________________________________________________________
conv1d_12 (Conv1D)              (None, 31, 256)      175872      input_5[0][0]                    
__________________________________________________________________________________________________
conv1d_15 (Conv1D)              (None, 1000, 256)    351744      input_6[0][0]                    
____________________________________________________________________________________________

Epoch 69/100
117/117 [==============================] - 61s 520ms/step - loss: 0.0043 - accuracy: 0.9991 - val_loss: 0.4297 - val_accuracy: 0.9268
Epoch 70/100
117/117 [==============================] - 65s 552ms/step - loss: 0.0042 - accuracy: 0.9988 - val_loss: 0.4504 - val_accuracy: 0.9238
Epoch 71/100
117/117 [==============================] - 65s 553ms/step - loss: 0.0045 - accuracy: 0.9986 - val_loss: 0.4275 - val_accuracy: 0.9278
Epoch 72/100
117/117 [==============================] - 64s 543ms/step - loss: 0.0038 - accuracy: 0.9989 - val_loss: 0.5048 - val_accuracy: 0.9215
Epoch 73/100
117/117 [==============================] - 65s 556ms/step - loss: 0.0030 - accuracy: 0.9994 - val_loss: 0.4787 - val_accuracy: 0.9248
Epoch 74/100
117/117 [==============================] - 65s 552ms/step - loss: 0.0041 - accuracy: 0.9988 - val_loss: 0.4780 - val_accuracy: 0.9249
Epoch 75/100
117/117 [==============================] - 65s 551ms/step - loss: 0.0028 - accuracy: 0.9992 - val_loss: 0

In [ ]:
################################################################################################################################################
#Train the traits only (OU) network
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
train_traits_ds = train_ds.map(lambda inputs, y: (inputs[0], y))
val_traits_ds   = val_ds.map(lambda inputs, y: (inputs[0], y))

# train the network
model5 = OU_subset(traits_all_mm, train_traits_ds, val_traits_ds)

          
model5.save(filepath='./TrainedModels/Trained_Traits_Model_OU.acc.mod')

Model: "model_5"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         [(None, 31, 229)]         0         
_________________________________________________________________
conv1d_6 (Conv1D)            (None, 31, 256)           175872    
_________________________________________________________________
batch_normalization_6 (Batch (None, 31, 256)           1024      
_________________________________________________________________
conv1d_7 (Conv1D)            (None, 31, 256)           196608    
_________________________________________________________________
batch_normalization_7 (Batch (None, 31, 256)           1024      
_________________________________________________________________
conv1d_8 (Conv1D)            (None, 31, 256)           196608    
_________________________________________________________________
batch_normalization_8 (Batch (None, 31, 256)           1024

Epoch 42/100
117/117 [==============================] - 62s 534ms/step - loss: 0.0451 - accuracy: 0.9850 - val_loss: 0.3494 - val_accuracy: 0.9181
Epoch 43/100
117/117 [==============================] - 62s 529ms/step - loss: 0.0419 - accuracy: 0.9869 - val_loss: 0.3503 - val_accuracy: 0.9191
Epoch 44/100
117/117 [==============================] - 62s 530ms/step - loss: 0.0372 - accuracy: 0.9880 - val_loss: 0.3576 - val_accuracy: 0.9180
Epoch 45/100
117/117 [==============================] - 62s 530ms/step - loss: 0.0391 - accuracy: 0.9876 - val_loss: 0.3724 - val_accuracy: 0.9177
Epoch 46/100
117/117 [==============================] - 64s 545ms/step - loss: 0.0356 - accuracy: 0.9886 - val_loss: 0.3731 - val_accuracy: 0.9184
Epoch 47/100
117/117 [==============================] - 63s 541ms/step - loss: 0.0347 - accuracy: 0.9888 - val_loss: 0.3717 - val_accuracy: 0.9184
Epoch 48/100
117/117 [==============================] - 63s 534ms/step - loss: 0.0327 - accuracy: 0.9897 - val_loss: 0

Epoch 98/100
117/117 [==============================] - 63s 538ms/step - loss: 0.0100 - accuracy: 0.9971 - val_loss: 0.5466 - val_accuracy: 0.9200
Epoch 99/100
117/117 [==============================] - 62s 530ms/step - loss: 0.0115 - accuracy: 0.9969 - val_loss: 0.5413 - val_accuracy: 0.9207
Epoch 100/100
117/117 [==============================] - 64s 545ms/step - loss: 0.0114 - accuracy: 0.9965 - val_loss: 0.5605 - val_accuracy: 0.9194
Time: 6266.836181163788
INFO:tensorflow:Assets written to: ./TrainedModels/Trained_Traits_Model_OU.acc.mod/assets


In [6]:
################################################################################################################################################
#Perform cross-validation
################################################################################################################################################


model1 = load_model('./TrainedModels/Trained_Comb_Model_1KSNPs_BM.acc.mod')
model2 = load_model('./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod')
model3 = load_model('./TrainedModels/Trained_Traits_Model_BM.acc.mod')
model4 = load_model('./TrainedModels/Trained_Comb_Model_1KSNPs_OU.acc.mod')
model5 = load_model('./TrainedModels/Trained_Traits_Model_OU.acc.mod')

traits_BM = []
traits_BM = np.loadtxt("./testSims/traits/traits_BM.txt").reshape(4000,-1,28)

for i in range(traits_BM.shape[2]):
    traits_BM[:, :, i] = scalers_BM[i].transform(traits_BM[:, :, i]) 

traits_BM = np.array(traits_BM)

#Add missing individuals
for i in range(traits_BM.shape[0]):
  for j in range(0,4):
    traits_BM[i,j,:] = 0
  for j in range(77,81):
    traits_BM[i,j,:] = 0
  for j in range(134,136):
    traits_BM[i,j,:] = 0

d_traits = []
d_traits = np.loadtxt("./testSims/traits/traits_disc.txt").reshape(4000,-1,3)
traits = np.concatenate((traits_BM,d_traits),axis=2)

u1 = np.load("./testSims/Model_2sp.npz")
u2 = np.load("./testSims/Model_6sp.npz")
u3 = np.load("./testSims/Model_6spMig.npz")
u4 = np.load("./testSims/Model_6spMig_all.npz")

xtest=np.concatenate((u1['Model_2sp'],u2['Model_6sp'],u3['Model_6spMig'],u4['Model_6spMig_all']),axis=0)

#transform major alleles in -1 and minor 1
for arr,array in enumerate(xtest):
  for idx,row in enumerate(array):
    if np.count_nonzero(row) > len(row)/2:
      xtest[arr][idx][xtest[arr][idx] == 1] = -1
      xtest[arr][idx][xtest[arr][idx] == 0] = 1
    else:
      xtest[arr][idx][xtest[arr][idx] == 0] = -1

#Add missing data as 0s, according to a specifies missing data percentage
#54,477,268 SNP datapoints and 24,326,002 missing genotypes = 44.6%
missD_perc = 44.6
missD = int(xtest.shape[1]*xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
  for m in range(missD):
    j = random.randint(0, xtest.shape[1] - 1)
    k = random.randint(0, xtest.shape[2] - 1)
    xtest[i][j][k] = 0


ytest=[0 for i in range(len(u1['Model_2sp']))]
ytest.extend([1 for i in range(len(u2['Model_6sp']))])
ytest.extend([2 for i in range(len(u3['Model_6spMig']))])
ytest.extend([3 for i in range(len(u4['Model_6spMig_all']))])
ytest = np.array(ytest)

#first get the predictions
pred = model1.predict([np.swapaxes(traits,1,2), xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))


#first get the predictions
pred = model2.predict(xtest)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model3.predict(np.swapaxes(traits,1,2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

traits_OU = []
traits_OU = np.loadtxt("./testSims/traits/traits_OU.txt").reshape(4000,-1,28)

for i in range(traits_OU.shape[2]):
    traits_OU[:, :, i] = scalers_OU[i].transform(traits_OU[:, :, i]) 

traits_OU = np.array(traits_OU)

#Add missing individuals
for i in range(traits_OU.shape[0]):
  for j in range(0,4):
    traits_OU[i,j,:] = 0
  for j in range(77,81):
    traits_OU[i,j,:] = 0
  for j in range(134,136):
    traits_OU[i,j,:] = 0

d_traits = []
d_traits = np.loadtxt("./testSims/traits/traits_disc.txt").reshape(4000,-1,3)
traits = np.concatenate((traits_OU,d_traits),axis=2)

#first get the predictions
pred = model4.predict([np.swapaxes(traits,1,2), xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))


#first get the predictions
pred = model5.predict(np.swapaxes(traits,1,2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

[[1000    0    0    0]
 [   0  989   11    0]
 [   4  216  728   52]
 [   2    1   76  921]]
[[1000    0    0    0]
 [   0 1000    0    0]
 [   0  456  544    0]
 [   0   25  475  500]]
[[990   0   0  10]
 [  1 867 132   0]
 [  1  74 840  85]
 [  1   0  42 957]]
[[1000    0    0    0]
 [   0  962   38    0]
 [   1  137  810   52]
 [  15    0   82  903]]
[[995   0   0   5]
 [  0 866 131   3]
 [  0 109 770 121]
 [  2   0  41 957]]


In [ ]:
################################################################################################################################################
#Predict empirical data
################################################################################################################################################

# Load the SNP data set and resample it to build 100 bootstrap replicates
inSNPs=np.loadtxt("./input_T.txt")
inp=np.array(inSNPs)
num_samples=100
res = []
for i in range(0,num_samples):
	idx = np.random.choice(inp.shape[0], 1000, replace=False)
	n = inp[idx,:]
	res.append(np.array(n))

EmpSNPs = np.array(res)

#transform major alleles in -1 and minor 1
for arr,array in enumerate(EmpSNPs):
  for idx,row in enumerate(array):
    if np.count_nonzero(row==1) > np.count_nonzero(row==-1):
      EmpSNPs[arr][idx][EmpSNPs[arr][idx] == 1] = 9
      EmpSNPs[arr][idx][EmpSNPs[arr][idx] == -1] = 1
      EmpSNPs[arr][idx][EmpSNPs[arr][idx] == 9] = -1

#load the trait data
Laqu_Lpel=np.genfromtxt("./input_traits_Laqu_Lpel.txt", delimiter="\t", filling_values=0)
Laqu=np.genfromtxt("./input_traits_Laqu.txt", delimiter="\t", filling_values=0)
Lmeg=np.genfromtxt("./input_traits_Lmeg.txt", delimiter="\t", filling_values=0)
Loua=np.genfromtxt("./input_traits_Loua.txt", delimiter="\t", filling_values=0)
Lozk_Lmeg=np.genfromtxt("./input_traits_Lozk_Lmeg.txt", delimiter="\t", filling_values=0)
Lozk=np.genfromtxt("./input_traits_Lozk.txt", delimiter="\t", filling_values=0)
Lpel_Lmeg=np.genfromtxt("./input_traits_Lpel_Lmeg.txt", delimiter="\t", filling_values=0)
Lpel=np.genfromtxt("./input_traits_Lpel.txt", delimiter="\t", filling_values=0)
Lsol_Lmeg=np.genfromtxt("./input_traits_Lsol_Lmeg.txt", delimiter="\t", filling_values=0)
Lsol=np.genfromtxt("./input_traits_Lsol.txt", delimiter="\t", filling_values=0)

Laqu_Lpel=np.array(Laqu_Lpel)
Laqu=np.array(Laqu)
Lmeg=np.array(Lmeg)
Loua=np.array(Loua)
Lozk_Lmeg=np.array(Lozk_Lmeg)
Lozk=np.array(Lozk)
Lpel_Lmeg=np.array(Lpel_Lmeg)
Lpel=np.array(Lpel)
Lsol_Lmeg=np.array(Lsol_Lmeg)
Lsol=np.array(Lsol)

#resample the trait data to build 100 bootstrap replicates
res = []
for i in range(0,num_samples):
  n = np.zeros((4,31))
  idx_aqu_pel = np.random.choice(Laqu_Lpel.shape[0], 6, replace=False)
  n = np.concatenate((n,Laqu_Lpel[idx_aqu_pel,:]), axis=0)
  idx_aqu = np.random.choice(Laqu.shape[0], 68, replace=False)
  n = np.concatenate((n,Laqu[idx_aqu,:]), axis=0)
  n = np.concatenate((n,np.zeros((3,31))), axis=0)
  idx_meg = np.random.choice(Lmeg.shape[0], 44, replace=False)
  n = np.concatenate((n,Lmeg[idx_meg,:]), axis=0)
  idx_oua = np.random.choice(Loua.shape[0], 10, replace=False)
  n = np.concatenate((n,Loua[idx_oua,:]), axis=0)
  n = np.concatenate((n,np.zeros((1,31))), axis=0)
  idx_ozk_meg = np.random.choice(Lozk_Lmeg.shape[0], 8, replace=False)
  n = np.concatenate((n,Lozk_Lmeg[idx_ozk_meg,:]), axis=0)
  idx_ozk = np.random.choice(Lozk.shape[0], 15, replace=False)
  n = np.concatenate((n,Lozk[idx_ozk,:]), axis=0)
  idx_pel_meg = np.random.choice(Lpel_Lmeg.shape[0], 10, replace=False)
  n = np.concatenate((n,Lpel_Lmeg[idx_pel_meg,:]), axis=0)
  idx_pel = np.random.choice(Lpel.shape[0], 19, replace=False)
  n = np.concatenate((n,Lpel[idx_pel,:]), axis=0)
  idx_sol_meg = np.random.choice(Lsol_Lmeg.shape[0], 5, replace=False)
  n = np.concatenate((n,Lsol_Lmeg[idx_sol_meg,:]), axis=0)
  idx_sol = np.random.choice(Lsol.shape[0], 36, replace=False)
  n = np.concatenate((n,Lsol[idx_sol,:]), axis=0)
  res.append(np.array(n))

traits = np.array(res)

c_traits = traits[:,:,0:28]
d_traits = traits[:,:,28:32]

#Use standard scaling for the continuous traits.
for i in range(c_traits.shape[2]):
    c_traits[:, :, i] = scalers_BM[i].transform(c_traits[:, :, i]) 

emp_traits_BM = np.concatenate((c_traits, d_traits), axis=2)

c_traits = traits[:,:,0:28]

#Use standard scaling for the continuous traits.
for i in range(c_traits.shape[2]):
    c_traits[:, :, i] = scalers_OU[i].transform(c_traits[:, :, i]) 

emp_traits_OU = np.concatenate((c_traits, d_traits), axis=2)

In [19]:
Emp_Comb_BM_pred = model1.predict([np.swapaxes(emp_traits_BM,1,2),EmpSNPs])

np.savetxt("Pred_Emp_Comb_BM_Predictions.txt", Emp_Comb_BM_pred)

Emp_SNP_pred = model2.predict(EmpSNPs)

np.savetxt("Pred_Emp_SNP_Predictions.txt", Emp_SNP_pred)

Emp_traits_BM_pred = model3.predict(np.swapaxes(emp_traits_BM,1,2))

np.savetxt("Pred_Emp_traits_BM_Predictions.txt", Emp_traits_BM_pred)

Emp_Comb_OU_pred = model4.predict([np.swapaxes(emp_traits_OU,1,2),EmpSNPs])

np.savetxt("Pred_Emp_Comb_OU_Predictions.txt", Emp_Comb_OU_pred)

Emp_traits_OU_pred = model5.predict(np.swapaxes(emp_traits_OU,1,2))

np.savetxt("Pred_Emp_traits_OU_Predictions.txt", Emp_traits_OU_pred)